# 1. Đánh giá Khử trùng lặp sâu Tin tuyển dụng (Deduplication)
Notebook này thực hiện so khớp và đánh giá thuật toán khử trùng lặp theo 2 kịch bản chính:
1. **Khử trùng lặp cùng nguồn (Same-Source Deduplication):** So khớp các tin đăng trùng lặp từ cùng một trang tuyển dụng.
2. **Khử trùng lặp đa nguồn (Cross-Source Deduplication):** So khớp các tin tuyển dụng giống nhau nhưng đăng chéo nguồn (ví dụ: đăng trên cả ITviec và VietnamWorks).


## Bước 1.1: Thiết lập môi trường và nạp module


In [9]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''
import sys, json, importlib.util
from pathlib import Path
from dotenv import load_dotenv

workspace_root = Path(r'f:\HCMUS_KH\LuanVan\JobVisualization_BE')
script_dir = workspace_root / 'KiemThu' / 'KiemThu_Deduplication'
import_script_path = workspace_root / 'Db' / 'pipeline' / 'import' / '3_import' / 'import.py'

# Nap dong module import.py
spec = importlib.util.spec_from_file_location('import_module', str(import_script_path))
imp_mod = importlib.util.module_from_spec(spec)
sys.path.insert(0, str(import_script_path.parent))
sys.path.insert(0, str(workspace_root))
spec.loader.exec_module(imp_mod)

# Nap .env
load_dotenv(workspace_root / 'Db' / '.env', override=True)
db_url = os.environ.get('DATABASE_URL')
if not db_url:
    raise ValueError('DATABASE_URL chua duoc cau hinh trong Db/.env')
print('✓ Đã nạp thành công module import.py và cấu hình database!')


✓ Đã nạp thành công module import.py và cấu hình database!


## Bước 1.2: Truy vấn JDs thực tế và sinh các cặp chéo nguồn (Cross-Source)


In [10]:
import psycopg2
from collections import Counter

conn = psycopg2.connect(db_url)
cur = conn.cursor()

# Chon 2 cong ty co nhieu vi tri va JD sach de test Same-Source (FPT Software va Techcombank)
top_companies = [
    (357, 'FPT Software'),
    (546, 'Techcombank')
]
print('Các công ty được chọn để test trùng lặp cùng nguồn (Same-Source):')
for cid, name in top_companies:
    print(f'  - [{cid}] {name}')

selected_jobs = []
for company_id, company_name in top_companies:
    cur.execute('''
        SELECT job_id, company_id, title, description, job_posting_url, scraped_at, source_name
        FROM jobs
        WHERE company_id = %s
          AND description IS NOT NULL AND LENGTH(description) > 50
        ORDER BY scraped_at DESC
        LIMIT 6
    ''', (company_id,))
    for row in cur.fetchall():
        selected_jobs.append({
            'job_id': row[0],
            'company_id': row[1],
            'company_name': company_name,
            'title': row[2],
            'description': row[3],
            'url': row[4],
            'scraped_at': str(row[5]),
            'source_name': row[6] or 'unknown'
        })

# 2. Lay dung cap chéo nguon thuc te tu CSDL (Cong Ty Co Phan Sua Quoc Te Lof - ID 552)
print('\nNạp cặp trùng lặp chéo nguồn thực tế (ITviec vs VietnamWorks) của Sữa Quốc Tế Lof...')
cur.execute('''
    SELECT job_id, company_id, title, description, job_posting_url, scraped_at, source_name
    FROM jobs
    WHERE job_id IN (13489, 14725)
''')
for row in cur.fetchall():
    selected_jobs.append({
        'job_id': row[0],
        'company_id': row[1],
        'company_name': 'Công Ty Cổ Phần Sữa Quốc Tế Lof',
        'title': row[2],
        'description': row[3],
        'url': row[4],
        'scraped_at': str(row[5]),
        'source_name': row[6] or 'unknown'
    })

# 3. Gia lap va chen them cac tin trung lap đa nguon (Cross-Source)
print('Tạo thêm tin tuyển dụng giả lập đa nguồn (Cross-Source) trong bộ nhớ...')
synthetic_jobs = []
origs_to_dup = [j for j in selected_jobs if j['company_id'] != 552][:3]
for idx, orig in enumerate(origs_to_dup):
    dup = orig.copy()
    dup['job_id'] = orig['job_id'] + 50000
    dup['url'] = orig['url'] + f'?test_cross_source_dup={idx}'
    dup['title'] = orig['title'] + ' (Cross-Source Test)'
    # Doi nguon de dam bao day la cap da nguon (Cross-source)
    dup['source_name'] = 'vietnamworks' if orig['source_name'] == 'itviec' else 'itviec'
    
    if idx % 2 == 0:
        dup['description'] = orig['description']
        print(f'  -> Tạo trùng lặp chéo nguồn 100% cho Job ID: {orig["job_id"]} ({orig["source_name"]} -> {dup["source_name"]})')
    else:
        dup['description'] = orig['description'] + '\n\nApply now via our direct portal! Contact recruitment team.'
        print(f'  -> Tạo trùng lặp chéo nguồn gần đúng (~95%) cho Job ID: {orig["job_id"]} ({orig["source_name"]} -> {dup["source_name"]})')
    synthetic_jobs.append(dup)

selected_jobs.extend(synthetic_jobs)

# Luu dataset
dataset_path = script_dir / 'job_descriptions_dataset.json'
with open(dataset_path, 'w', encoding='utf-8') as f:
    json.dump(selected_jobs, f, ensure_ascii=False, indent=2)

print(f'✓ Tổng cộng có {len(selected_jobs)} jobs phục vụ kịch bản kiểm thử.')
cur.close()
conn.close()


Các công ty được chọn để test trùng lặp cùng nguồn (Same-Source):
  - [357] FPT Software
  - [546] Techcombank

Nạp cặp trùng lặp chéo nguồn thực tế (ITviec vs VietnamWorks) của Sữa Quốc Tế Lof...
Tạo thêm tin tuyển dụng giả lập đa nguồn (Cross-Source) trong bộ nhớ...
  -> Tạo trùng lặp chéo nguồn 100% cho Job ID: 14795 (itviec -> vietnamworks)
  -> Tạo trùng lặp chéo nguồn gần đúng (~95%) cho Job ID: 13885 (vietnamworks -> itviec)
  -> Tạo trùng lặp chéo nguồn 100% cho Job ID: 13599 (itviec -> vietnamworks)
✓ Tổng cộng có 17 jobs phục vụ kịch bản kiểm thử.


## Bước 1.3: Chạy thuật toán so khớp Cosine Similarity và Phân loại kịch bản


In [11]:
import math, re, datetime
from collections import Counter as _Counter

THRESHOLD    = 0.8
MIN_DESC_LEN = 50

def tfidf_cosine_sim(desc_a, desc_b):
    if not desc_a or len(desc_a.strip()) < MIN_DESC_LEN:
        return 0.0
    if not desc_b or len(desc_b.strip()) < MIN_DESC_LEN:
        return 0.0
    token_pattern = re.compile(r'\w+')
    tf_a = _Counter(token_pattern.findall(desc_a.lower()))
    tf_b = _Counter(token_pattern.findall(desc_b.lower()))
    all_words = set(tf_a.keys()) | set(tf_b.keys())
    sq_a = sq_b = dot = 0.0
    for w in all_words:
        in_a, in_b = w in tf_a, w in tf_b
        if in_a and in_b:
            va, vb = tf_a[w] * 1.0, tf_b[w] * 1.0
            dot += va * vb; sq_a += va**2; sq_b += vb**2
        elif in_a:
            sq_a += (tf_a[w] * 1.405465108) ** 2
        else:
            sq_b += (tf_b[w] * 1.405465108) ** 2
    la, lb = math.sqrt(sq_a), math.sqrt(sq_b)
    return dot / (la * lb) if la > 0 and lb > 0 else 0.0

# Nhom theo company_id
from collections import defaultdict
groups = defaultdict(list)
for job in selected_jobs:
    groups[job['company_id']].append(job)

all_pairs = []
now = datetime.datetime.now()
window_30d = datetime.timedelta(days=30)

for company_id, jobs_list in groups.items():
    for i in range(len(jobs_list)):
        for j in range(i + 1, len(jobs_list)):
            ja, jb = jobs_list[i], jobs_list[j]
            
            try:
                scraped_a = datetime.datetime.fromisoformat(ja['scraped_at'])
                scraped_b = datetime.datetime.fromisoformat(jb['scraped_at'])
                within_30d = (now - scraped_a <= window_30d) or (now - scraped_b <= window_30d)
            except Exception:
                within_30d = None
                
            sim = tfidf_cosine_sim(ja['description'], jb['description'])
            predicted = 'DUPLICATE' if sim > THRESHOLD else 'INDEPENDENT'
            
            # Xac dinh nguon
            is_cross_source = (ja['source_name'] != jb['source_name'])
            scenario = 'CROSS_SOURCE' if is_cross_source else 'SAME_SOURCE'
            
            # Tu dong xac dinh Ground Truth (Nhãn dung thuc te)
            is_synthetic_dup = (abs(ja['job_id'] - jb['job_id']) == 50000)
            is_real_lof_dup = (ja['job_id'] in (13489, 14725) and jb['job_id'] in (13489, 14725))
            is_same_url = (ja['url'] == jb['url'])
            
            if is_synthetic_dup or is_real_lof_dup or is_same_url:
                true_label = 'DUPLICATE'
            else:
                # Cac cap khac nhau trong DB
                true_label = 'INDEPENDENT'
                
            all_pairs.append({
                'job_id_a': ja['job_id'],
                'job_id_b': jb['job_id'],
                'company_id': company_id,
                'company_name': ja['company_name'],
                'title_a': ja['title'],
                'title_b': jb['title'],
                'source_a': ja['source_name'],
                'source_b': jb['source_name'],
                'url_a': ja['url'],
                'url_b': jb['url'],
                'similarity': round(sim, 4),
                'within_30_days': within_30d,
                'predicted_label': predicted,
                'true_label': true_label,
                'scenario': scenario
            })

all_pairs.sort(key=lambda x: x['similarity'], reverse=True)
print(f'Tổng số cặp tin đối soát: {len(all_pairs)}')
print(f'  - Cùng nguồn (Same-Source): {sum(1 for p in all_pairs if p["scenario"] == "SAME_SOURCE")}')
print(f'  - Đa nguồn (Cross-Source) : {sum(1 for p in all_pairs if p["scenario"] == "CROSS_SOURCE")}')


Tổng số cặp tin đối soát: 52
  - Cùng nguồn (Same-Source): 17
  - Đa nguồn (Cross-Source) : 35


## Bước 1.4: Lưu kết quả Ground Truth và Actual


In [12]:
gt_path = script_dir / 'ground_truth_deduplication.json'
with open(gt_path, 'w', encoding='utf-8') as f:
    json.dump(all_pairs, f, ensure_ascii=False, indent=2)

print('✓ Đã tạo file ground_truth_deduplication.json thành công!')
print('\n--- Xem thử 10 cặp có độ tương đồng cao nhất ---')
print(f'{"STT":<4} {"Sim":<6} {"Kịch bản":<15} {"Nguồn A":<12} {"Nguồn B":<12} {"Kết quả":<10}')
print('-'*70)
for idx, p in enumerate(all_pairs[:10], 1):
    print(f'{idx:<4} {p["similarity"]:<6} {p["scenario"]:<15} {p["source_a"]:<12} {p["source_b"]:<12} {p["true_label"]:<10}')


✓ Đã tạo file ground_truth_deduplication.json thành công!

--- Xem thử 10 cặp có độ tương đồng cao nhất ---
STT  Sim    Kịch bản        Nguồn A      Nguồn B      Kết quả   
----------------------------------------------------------------------
1    1.0    CROSS_SOURCE    itviec       vietnamworks DUPLICATE 
2    1.0    CROSS_SOURCE    itviec       vietnamworks DUPLICATE 
3    0.9913 CROSS_SOURCE    vietnamworks itviec       DUPLICATE 
4    0.9684 CROSS_SOURCE    itviec       vietnamworks DUPLICATE 
5    0.6246 SAME_SOURCE     linkedin     linkedin     INDEPENDENT
6    0.6165 SAME_SOURCE     itviec       itviec       INDEPENDENT
7    0.46   SAME_SOURCE     itviec       itviec       INDEPENDENT
8    0.46   CROSS_SOURCE    itviec       vietnamworks INDEPENDENT
9    0.4459 CROSS_SOURCE    vietnamworks itviec       INDEPENDENT
10   0.4433 CROSS_SOURCE    vietnamworks itviec       INDEPENDENT
